In [1]:
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
import numpy as np

def cosine_similarity(a, b):
    a_v = np.array(a)
    b_v = np.array(b)
    dot_product = np.dot(a_v, b_v)

    magnitude_a = np.linalg.norm(a)
    magnitude_b = np.linalg.norm(b)

    return dot_product / (magnitude_a * magnitude_b)


dog = [0.8 , 0.6 , 0.3]
cat = [0.75, 0.65, 0.35]
man = [-0.5 , 0.2 , 0.9]

dog_man = cosine_similarity(dog, man)
dog_cat = cosine_similarity(dog, cat)
cat_man = cosine_similarity(cat, man)


print(dog_man)
print(dog_cat)
print(cat_man)

-0.00913251529954447
0.9966186334192181
0.06342052458503866


In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2",
)

c:\Users\akash\OneDrive\Desktop\Rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3029.91it/s]


In [4]:
vector = embeddings.embed_query("I love coding")
print(vector)

[-0.055323339998722076, -0.017197521403431892, -0.030153872445225716, -0.02548508159816265, 0.04150587320327759, -0.04072405770421028, 0.03350682929158211, 0.011637558229267597, 0.022708144038915634, 0.065151147544384, -0.024583684280514717, 0.013861224055290222, 0.018221663311123848, 0.0076222652569413185, 0.017752034589648247, -0.0077478704042732716, -0.09681794047355652, 0.005888055078685284, -0.021910395473241806, -0.07908615469932556, -0.10790747404098511, -0.0405949130654335, -0.008594430051743984, -0.014864321798086166, 0.05945075675845146, 0.10874738544225693, 0.003004744416102767, -0.04015112668275833, 0.05152805522084236, -0.08490431308746338, -0.0711793527007103, 0.07259964942932129, 0.1382119357585907, 0.06595804542303085, 0.01580083929002285, 0.0361771434545517, 0.060219164937734604, -0.06931641697883606, -0.024434411898255348, 0.009563232772052288, -0.1432448923587799, 0.036074280738830566, 0.045089125633239746, -0.009268192574381828, -0.0039503430016338825, -0.0388667210

In [9]:
len(vector)

384

In [5]:

documents = [
    "I love learning Python programming.",
    "I enjoy writing code using Python.",

    "Machine learning helps computers learn from data.",
    "ML allows computers to identify patterns from data.",

    "Spring Boot is used to build Java backend applications.",
    "Java developers use Spring Boot to develop backend services.",

    "PostgreSQL is a relational database management system.",
    "PostgreSQL stores structured data in relational tables.",

    "Redis is an in-memory database used for caching.",
    "Redis stores data in memory to provide fast access.",
]


In [6]:
doc_embeddings = embeddings.embed_documents(documents)

In [7]:
len(doc_embeddings)

10

In [21]:
query = "Best framework for develope backend application using Java"
# query = "Best database for caching"
# query = "Best database for storing data like in tables"
query_vector = embeddings.embed_query(query)

In [19]:
scores = []
for i , vec in enumerate(doc_embeddings):
    scores.append({
        "document":documents[i],
        "score":cosine_similarity(query_vector , vec)
    })


    

In [20]:
scores.sort(key=lambda x: x['score'],reverse=True)

for ele in scores:
    print(ele["document"])
    print(ele["score"])
    print()


PostgreSQL stores structured data in relational tables.
0.5469955138627058

PostgreSQL is a relational database management system.
0.38841868648831324

Redis stores data in memory to provide fast access.
0.295528341228786

Redis is an in-memory database used for caching.
0.2652936369544553

ML allows computers to identify patterns from data.
0.2102047588673955

Machine learning helps computers learn from data.
0.16570003950299456

I enjoy writing code using Python.
0.13193045326045036

I love learning Python programming.
0.0937864519223763

Spring Boot is used to build Java backend applications.
0.0807924916319524

Java developers use Spring Boot to develop backend services.
0.0591763650992376



## `Vector Store`

In [22]:
from langchain_core.vectorstores import InMemoryVectorStore

In [23]:
vector_store = InMemoryVectorStore.from_texts(
    texts=documents,
    embedding=embeddings
)

In [27]:
chunks = vector_store.similarity_search(query)
chunks

[Document(id='be9e4cd1-1cec-4f47-b1c6-d8740fef0123', metadata={}, page_content='Java developers use Spring Boot to develop backend services.'),
 Document(id='a726d9c9-d8f1-4588-bb1e-e79777f4bd02', metadata={}, page_content='Spring Boot is used to build Java backend applications.'),
 Document(id='a5062688-d182-44fa-9bd5-fdfb05c4d9d0', metadata={}, page_content='I enjoy writing code using Python.'),
 Document(id='9a8f8f96-35f3-4744-84ae-82983a4280cf', metadata={}, page_content='PostgreSQL is a relational database management system.')]

# `Cohere embeddings`

In [35]:
from langchain_cohere import CohereEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [63]:
loader = PyMuPDFLoader(file_path="../data/Profile.pdf")

In [64]:
data = loader.load()
len(data)

2

In [65]:
spliter = RecursiveCharacterTextSplitter(chunk_size=250,chunk_overlap=40)
chunks = spliter.split_documents(data)
len(chunks)

19

In [66]:
cohere_embeddings = CohereEmbeddings(
    model="embed-english-v3.0",
)

In [67]:
vector_store = InMemoryVectorStore.from_documents(
    embedding=cohere_embeddings,
    documents=chunks
)

In [69]:
query = "Where Akash have worked as a Intern"
chunk_results = vector_store.similarity_search(query,k=2)



for chunk in chunk_results:
    print(chunk_results[0].metadata)
    print()
    print(chunk_results[0].page_content)
    print("\n\n")


{'producer': 'Apache FOP Version 2.3', 'creator': '', 'creationdate': '2026-10-27T06:22:40+00:00', 'source': '../data/Profile.pdf', 'file_path': '../data/Profile.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Resume', 'author': 'LinkedIn', 'subject': 'Resume generated from profile', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': 'D:20261027062240Z', 'page': 1}

Experience
Appreciate
Software Engineer Intern
August 2025 - July 2026 (1 year)
Noida, Uttar Pradesh, India
- Developed Flutter frontend modules and backend services using FastAPI and
Spring Boot for fintech and digital wealth management products.



{'producer': 'Apache FOP Version 2.3', 'creator': '', 'creationdate': '2026-10-27T06:22:40+00:00', 'source': '../data/Profile.pdf', 'file_path': '../data/Profile.pdf', 'total_pages': 2, 'format': 'PDF 1.4', 'title': 'Resume', 'author': 'LinkedIn', 'subject': 'Resume generated from profile', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate'